# V-JEPA 2 on HATREC — leakage-safe benchmark

Frozen V-JEPA 2 embeddings plus a linear probe. The primary split holds out complete cycles. A random-clip split is shown only to quantify leakage risk. Full 64-frame coverage is compared with 14 visible frames. Filenames are never passed to the model.


In [ ]:
%pip install -q -U "transformers>=5.0.0" accelerate decord kagglehub


In [ ]:
import os,json,time,hashlib,shutil,pickle
from pathlib import Path
import numpy as np,pandas as pd,torch
from decord import VideoReader,cpu
from transformers import AutoModel,AutoVideoProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler,normalize
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix
from sklearn.model_selection import train_test_split
WORK=Path(os.environ.get('VJEPA_WORKDIR','/kaggle/working/vjepa_hatrec' if Path('/kaggle/working').exists() else './vjepa_hatrec')).resolve();WORK.mkdir(parents=True,exist_ok=True)
TASKS={0:'Assembling spring',1:'White plastic',2:'Screwing-1',3:'Inflating valve',4:'Black plastic',5:'Screwing-2',6:'Fixing cable'}


In [ ]:
# Locate attached/local data, otherwise fetch the public Kaggle dataset.
root=os.environ.get('HATREC_ROOT','').strip();roots=[Path(root)] if root else list(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []
videos=[]
for r in roots:
    if r.exists():videos+=list(r.rglob('Cycle_*_task_*.mp4'))
if not videos:
    import kagglehub
    videos=list(Path(kagglehub.dataset_download('ayoznur/hatrec-video-dataset')).rglob('Cycle_*_task_*.mp4'))
rows=[]
for p in sorted(set(videos)):
    parts=p.stem.split('_')
    if len(parts)==4:rows.append({'path':str(p.resolve()),'cycle':int(parts[1]),'label':int(parts[3]),'neutral_id':f'video_{len(rows)+1:04d}'})
df=pd.DataFrame(rows);assert len(df)==546 and df.cycle.nunique()==78 and sorted(df.label.unique())==list(range(7)),df.shape
# Parse ground truth once, then physically rename/copy media before any model access.
neutral_dir=WORK/'neutral_media';neutral_dir.mkdir(exist_ok=True);neutral_paths=[];hashes=[]
for row in df.itertuples():
    source=Path(row.path);target=neutral_dir/f'{row.neutral_id}.mp4';digest=hashlib.sha256(source.read_bytes()).hexdigest()
    if not target.exists() or target.stat().st_size!=source.stat().st_size:shutil.copyfile(source,target)
    assert hashlib.sha256(target.read_bytes()).hexdigest()==digest;neutral_paths.append(str(target));hashes.append(digest)
df['path']=neutral_paths;df['sha256']=hashes;assert all('task_' not in Path(p).name.lower() and 'cycle_' not in Path(p).name.lower() for p in df.path)
exact_dupes=int(df.sha256.duplicated().sum());assert exact_dupes==0
cycles=np.array(sorted(df.cycle.unique()));rng=np.random.default_rng(20260729);rng.shuffle(cycles)
train_cycles=set(cycles[:54]);val_cycles=set(cycles[54:66]);test_cycles=set(cycles[66:])
df['split']=np.where(df.cycle.isin(train_cycles),'train',np.where(df.cycle.isin(val_cycles),'val','test'))
split_report={'videos':len(df),'physical_filename_neutralization':True,'exact_duplicates':exact_dupes,'train_cycles':54,'val_cycles':12,'test_cycles':12,'rows':df.split.value_counts().to_dict()}
print(json.dumps(split_report,indent=2));display(df.drop(columns=['path','sha256']).head())


In [ ]:
from concurrent.futures import ThreadPoolExecutor
MODEL_ID='facebook/vjepa2-vitl-fpc64-256';assert torch.cuda.is_available(),'GPU required'
N_GPU=torch.cuda.device_count();assert N_GPU>=2,f'T4x2 required, found {N_GPU} GPU'
devices=[torch.device(f'cuda:{i}') for i in range(2)]
processors=[AutoVideoProcessor.from_pretrained(MODEL_ID) for _ in devices]
models=[]
for device in devices:
    models.append(AutoModel.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to(device).eval())
print('V-JEPA replicas:',[(str(d),torch.cuda.get_device_name(d)) for d in devices])
def video_tensor(path,n_visible):
    vr=VideoReader(path,ctx=cpu(0));base=np.unique(np.rint(np.linspace(0,len(vr)-1,min(n_visible,len(vr)))).astype(int));raw=vr.get_batch(base).asnumpy();idx=np.rint(np.linspace(0,len(raw)-1,64)).astype(int)
    return torch.from_numpy(raw[idx]).permute(0,3,1,2)
def embed(path,n_visible,model,processor,device):
    x=processor(video_tensor(path,n_visible),return_tensors='pt')['pixel_values_videos'].to(device,dtype=torch.float16)
    with torch.inference_mode():z=model.get_vision_features(x)
    return z.mean(dim=1).float().cpu().numpy()[0]
def extract(name,n_visible):
    def worker(gpu_id):
        torch.cuda.set_device(gpu_id);cache=WORK/f'embeddings_{name}_gpu{gpu_id}.pkl'
        saved=pickle.loads(cache.read_bytes()) if cache.exists() else {};shard=df.iloc[gpu_id::2]
        completed=0
        for _,row in shard.iterrows():
            if row.neutral_id not in saved:saved[row.neutral_id]=embed(row.path,n_visible,models[gpu_id],processors[gpu_id],devices[gpu_id])
            completed+=1
            if completed%25==0 or completed==len(shard):
                tmp=cache.with_suffix('.tmp');tmp.write_bytes(pickle.dumps(saved));tmp.replace(cache)
                print(name,f'GPU{gpu_id}',completed,'/',len(shard),flush=True)
        return saved
    with ThreadPoolExecutor(max_workers=2) as pool:parts=list(pool.map(worker,[0,1]))
    saved={};[saved.update(part) for part in parts]
    missing=[k for k in df.neutral_id if k not in saved];assert not missing,missing[:5]
    combined=WORK/f'embeddings_{name}.pkl';tmp=combined.with_suffix('.tmp');tmp.write_bytes(pickle.dumps(saved));tmp.replace(combined)
    return np.stack([saved[k] for k in df.neutral_id])
X64=extract('full64',64);X14=extract('visible14',14);print(X64.shape,X14.shape)


In [ ]:
def probe(X,tr,va,te):
    y=df.label.values;best=None
    for C in (.01,.1,1,10):
        clf=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=3000,class_weight='balanced'));clf.fit(X[tr],y[tr]);score=accuracy_score(y[va],clf.predict(X[va]))
        if best is None or score>best[0]:best=(score,C)
    fit=tr|va;clf=make_pipeline(StandardScaler(),LogisticRegression(C=best[1],max_iter=3000,class_weight='balanced'));clf.fit(X[fit],y[fit]);pred=clf.predict(X[te]);proba=clf.predict_proba(X[te])
    return {'C':best[1],'accuracy':accuracy_score(y[te],pred),'macro_f1':f1_score(y[te],pred,average='macro'),'truth':y[te],'pred':pred,'proba':proba}
tm=(df.split=='train').values;vm=(df.split=='val').values;xm=(df.split=='test').values
results={'cycle_holdout_full64':probe(X64,tm,vm,xm),'cycle_holdout_visible14':probe(X14,tm,vm,xm)}
idx=np.arange(len(df));tr,tmp=train_test_split(idx,test_size=.3,random_state=20260729,stratify=df.label);va,te=train_test_split(tmp,test_size=.5,random_state=20260729,stratify=df.label.values[tmp])
results['random_clip_full64']=probe(X64,np.isin(idx,tr),np.isin(idx,va),np.isin(idx,te))
def calibration(truth,proba,bins=10):
    conf=proba.max(1);pred=proba.argmax(1);ece=0.0
    for lo in np.linspace(0,1,bins,endpoint=False):
        mask=(conf>=lo)&(conf<lo+1/bins)
        if mask.any():ece+=mask.mean()*abs((pred[mask]==truth[mask]).mean()-conf[mask].mean())
    onehot=np.eye(proba.shape[1])[truth];return float(ece),float(np.mean(np.sum((proba-onehot)**2,axis=1)))
summary={k:{m:v[m] for m in ('C','accuracy','macro_f1')} for k,v in results.items()};summary['capability_mapping']={'B1':'task-phase/state classification; primary=macro_f1','B2':'not claimed: clips are pre-segmented','B10':'not claimed: held-out cycles are a generalization control, not a true OOD abstention set'};summary['chance_accuracy']=1/7;summary['split']=split_report
ece,brier=calibration(results['cycle_holdout_full64']['truth'],results['cycle_holdout_full64']['proba']);summary['cycle_holdout_full64'].update({'ece':ece,'multiclass_brier':brier})
# Mandatory B1 controls: majority-blind and shuffled-label probe.
truth=results['cycle_holdout_full64']['truth'];majority=np.full_like(truth,df.loc[tm|vm,'label'].mode()[0]);summary['blind_majority']={'accuracy':accuracy_score(truth,majority),'macro_f1':f1_score(truth,majority,average='macro')}
fit=tm|vm;ys=df.label.values.copy();ys[fit]=rng.permutation(ys[fit]);shuf=make_pipeline(StandardScaler(),LogisticRegression(C=results['cycle_holdout_full64']['C'],max_iter=3000,class_weight='balanced'));shuf.fit(X64[fit],ys[fit]);sp=shuf.predict(X64[xm]);summary['shuffled_label_control']={'accuracy':accuracy_score(truth,sp),'macro_f1':f1_score(truth,sp,average='macro')}
A=normalize(X64[tm]);B=normalize(X64[xm]);maxima=(B@A.T).max(axis=1);summary['test_to_train_cosine']={'mean_max':float(maxima.mean()),'p95_max':float(np.quantile(maxima,.95)),'above_0_999':int((maxima>.999).sum())}
print(json.dumps(summary,indent=2))


In [ ]:
primary=results['cycle_holdout_full64'];pred_df=df.loc[xm,['neutral_id','cycle','label']].copy();pred_df['prediction']=primary['pred'];pred_df['correct']=pred_df.label==pred_df.prediction
# Cluster bootstrap by held-out cycle, never by overlapping clips.
boot=[];test_cycle_ids=pred_df.cycle.unique();brng=np.random.default_rng(20260730)
for _ in range(2000):
    sampled=brng.choice(test_cycle_ids,len(test_cycle_ids),replace=True);parts=[pred_df[pred_df.cycle==c] for c in sampled];bdf=pd.concat(parts,ignore_index=True);boot.append(f1_score(bdf.label,bdf.prediction,average='macro'))
summary['cycle_bootstrap_macro_f1_95ci']=[float(np.quantile(boot,.025)),float(np.quantile(boot,.975))]
pred_df.to_csv(WORK/'hatrec_vjepa2_predictions.csv',index=False);(WORK/'hatrec_vjepa2_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
import matplotlib.pyplot as plt,seaborn as sns
fig,axes=plt.subplots(1,3,figsize=(18,5));pd.Series({k:v['accuracy']*100 for k,v in summary.items() if isinstance(v,dict) and 'accuracy' in v}).plot.bar(ax=axes[0],color=['#76b900','#4c78a8','#e15759']);axes[0].axhline(100/7,ls='--',c='black');axes[0].set_title('Split and frame ablations')
sns.heatmap(confusion_matrix(primary['truth'],primary['pred']),annot=True,fmt='d',cmap='Blues',ax=axes[1]);axes[1].set_title('Cycle-held-out confusion');axes[2].hist(maxima,bins=25,color='#f28e2b');axes[2].set_title('Max test-to-train cosine')
plt.tight_layout();plt.savefig(WORK/'hatrec_vjepa2_analysis.png',dpi=160,bbox_inches='tight');plt.show()
archive=shutil.make_archive(str(WORK.parent/'hatrec_vjepa2_artifacts'),'zip',WORK);print('RESULT',archive)
